In [1]:
# Imports.
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Loading the raw dataset and parse dates.
df = pd.read_csv("../data/raw/energydata_complete.csv", parse_dates=["date"])

# Setting 'date' as index and sorting the DataFrame.
df.set_index("date", inplace=True)
df.sort_index(inplace=True)

# ---------------------------
# Feature Engineering.
# ---------------------------

# Time-based features.
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month
df["weekend"] = df["dayofweek"].apply(lambda x: 1 if x >= 5 else 0)

# Lag features for target variable (Appliances).
for lag in range(1, 25):  # last 24 hours
    df[f"Appliances_lag_{lag}"] = df["Appliances"].shift(lag)

# Rolling window features (to capture trends).
df["Appliances_roll_mean_3"] = df["Appliances"].rolling(window=3).mean()
df["Appliances_roll_std_3"] = df["Appliances"].rolling(window=3).std()

df["Appliances_roll_mean_6"] = df["Appliances"].rolling(window=6).mean()
df["Appliances_roll_std_6"] = df["Appliances"].rolling(window=6).std()

# Interaction feature: Outdoor temperature * humidity
df["T_out_Hum_out"] = df["T_out"] * df["RH_out"]

# Dropping rows with NaNs from lag/rolling operations.
df.dropna(inplace=True)

# Saving processed data.
df.to_csv("../data/processed/energydata_features.csv")

# Print confirmation.
print("Feature engineering complete. Final dataset shape:", df.shape)
df.head()


Feature engineering complete. Final dataset shape: (19711, 61)


,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,Appliances_lag_20,Appliances_lag_21,Appliances_lag_22,Appliances_lag_23,Appliances_lag_24,Appliances_roll_mean_3,Appliances_roll_std_3,Appliances_roll_mean_6,Appliances_roll_std_6,T_out_Hum_out
date,,,,,,,,,,,,,,,,,,,,,
2016-01-11 21:00:00,110,30,21.133333,46.060000,20.426667,44.760000,20.29,46.433333,19.390000,48.193333,...,60.0,50.0,50.0,60.0,60.0,110.000000,0.000000,130.000000,31.622777,522.000000
2016-01-11 21:10:00,110,20,21.200000,45.800000,20.500000,44.760000,20.39,46.223333,19.390000,47.800000,...,50.0,60.0,50.0,50.0,60.0,110.000000,0.000000,125.000000,32.093613,517.188889
2016-01-11 21:20:00,100,30,21.290000,45.900000,20.533333,45.090000,20.39,46.090000,19.390000,47.560000,...,60.0,50.0,60.0,50.0,50.0,106.666667,5.773503,121.666667,33.714487,512.355556
2016-01-11 21:30:00,100,20,21.356667,45.826667,20.666667,45.163333,20.39,46.090000,19.390000,47.500000,...,60.0,60.0,50.0,60.0,50.0,103.333333,5.773503,106.666667,5.163978,507.500000
2016-01-11 21:40:00,100,20,21.390000,45.690000,20.700000,45.060000,20.39,46.090000,19.426667,47.993333,...,60.0,60.0,60.0,50.0,60.0,100.000000,0.000000,105.000000,5.477226,502.622222
